In [19]:
import numpy as np
import pandas as pd
import cv2
import requests

from pathlib import Path
from huggingface_hub import hf_hub_url, login
from huggingface_hub.utils import get_token

In [20]:
login()

In [5]:
# Project Configuration
PROJECT_DIR = Path("/content/nexar_project")
OUTPUT_DIR = PROJECT_DIR / "outputs"
PROCESSED_DIR = OUTPUT_DIR / "processed_data"

# Create output directories
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Temporal Configuration
OBSERVATION_WINDOW = 4.0
PREDICTION_HORIZON = 1.0
WINDOW_STRIDE = 1.0

# Multi-Scale Sampling Configuration
COARSE_FPS = 1
MEDIUM_FPS = 2
FINE_FPS = 4

# Image Configuration
IMAGE_SIZE = (224, 224)

print("Project configuration completed.")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Processed data directory: {PROCESSED_DIR}")

Project configuration completed.
Output directory: /content/nexar_project/outputs
Processed data directory: /content/nexar_project/outputs/processed_data


In [8]:
PROJECT_DIR = Path("/content")
SPLIT_FILE = PROJECT_DIR / "video_split.csv"

if not SPLIT_FILE.exists():
    raise FileNotFoundError(
        f"Video split file not found: {SPLIT_FILE}"
    )

video_split = pd.read_csv(SPLIT_FILE)
required_columns = {"file_name", "label", "split"}

if not required_columns.issubset(video_split.columns):
    raise ValueError("Missing required columns in video_split.csv")

train_videos = video_split[
    video_split["split"] == "train"
].copy()

val_videos = video_split[
    video_split["split"] == "validation"
].copy()

# Display dataset information
print("Video split loaded successfully.")
print(f"Total videos: {len(video_split)}")
print(f"Training videos: {len(train_videos)}")
print(f"Validation videos: {len(val_videos)}")

print("\nTraining class distribution:")
print(train_videos["label"].value_counts())

print("\nValidation class distribution:")
print(val_videos["label"].value_counts())

Video split loaded successfully.
Total videos: 1500
Training videos: 1200
Validation videos: 300

Training class distribution:
label
1    600
0    600
Name: count, dtype: int64

Validation class distribution:
label
0    150
1    150
Name: count, dtype: int64


In [9]:
print("Available columns:")
print(video_split.columns.tolist())

print("\nFirst 5 rows:")
display(video_split.head())

print("\nMissing values:")
print(video_split.isnull().sum())

Available columns:
['file_name', 'time_of_event', 'time_of_alert', 'light_conditions', 'weather', 'scene', 'time_to_accident', 'label', 'split']

First 5 rows:


,file_name,time_of_event,time_of_alert,light_conditions,weather,scene,time_to_accident,label,split
0,00780.mp4,19.967,16.9,Normal,Cloudy,Highway,NaN,1,train
1,00661.mp4,20.400,17.1,Normal,Cloudy,Highway,NaN,1,train
2,01888.mp4,NaN,NaN,Twilight,Cloudy,Highway,NaN,0,train
3,02084.mp4,NaN,NaN,Normal,Clear,Urban,NaN,0,train
4,01714.mp4,NaN,NaN,Normal,Clear,Sub-urban,NaN,0,train



Missing values:
file_name              0
time_of_event        750
time_of_alert        750
light_conditions       0
weather                2
scene                  0
time_to_accident    1500
label                  0
split                  0
dtype: int64


In [10]:
def generate_windows(
    video_duration,
    label,
    event_time=None,
    observation_window=OBSERVATION_WINDOW,
    stride=WINDOW_STRIDE,
    prediction_horizon=PREDICTION_HORIZON
):
    windows = []
    start_time = 0.0

    while start_time + observation_window <= video_duration:
        end_time = start_time + observation_window
        if label == 1:
            time_to_event = event_time - end_time

            if time_to_event <= 0:
                start_time += stride
                continue

            window_label = int(
                time_to_event <= prediction_horizon
            )

        else:
            time_to_event = None
            window_label = 0

        windows.append({
            "start_time": start_time,
            "end_time": end_time,
            "time_to_event": time_to_event,
            "label": window_label
        })

        start_time += stride

    return windows


print("Sliding window function is ready.")

Sliding window function is ready.


In [12]:
def get_video_duration(video_path):
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)

    cap.release()

    if fps <= 0:
        raise ValueError(f"Invalid FPS for video: {video_path}")

    duration = frame_count / fps

    return duration


print("Video duration function is ready.")

Video duration function is ready.


In [21]:
sample_video = train_videos[
    train_videos["label"] == 1
].iloc[0]

file_name = sample_video["file_name"]

video_url = hf_hub_url(
    repo_id="nexar-ai/nexar_collision_prediction",
    filename=f"train/positive/{file_name}",
    repo_type="dataset"
)

headers = {
    "Range": "bytes=0-1023"
}

token = get_token()

if token:
    headers["Authorization"] = f"Bearer {token}"

response = requests.get(
    video_url,
    headers=headers,
    stream=True,
    allow_redirects=True,
    timeout=30
)

print("Video:", file_name)
print("HTTP Status:", response.status_code)
print("Content-Range:", response.headers.get("Content-Range"))
print("Accept-Ranges:", response.headers.get("Accept-Ranges"))
print("Content-Length:", response.headers.get("Content-Length"))

if response.status_code == 206:
    print("Byte-range streaming is supported.")
elif response.status_code == 200:
    print("Server returned a normal response; range support needs further investigation.")
else:
    print("Request failed. Please check the response status.")

response.close()

Video: 00780.mp4
HTTP Status: 206
Content-Range: bytes 0-1023/13201618
Accept-Ranges: bytes
Content-Length: 1024
Byte-range streaming is supported.
